In [ ]:
import urllib.request

files = {
    "EDITIONS": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "WORKS": "https://openlibrary.org/data/ol_dump_works_latest.txt.gz",
    "AUTHORS": "https://openlibrary.org/data/ol_dump_authors_latest.txt.gz",
}

for name, url in files.items():
    print(f"\n--- {name} ---")
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            size = r.headers.get("Content-Length")
            print("HTTP :", r.status)
            print("Taille :", round(int(size)/1024**3, 2), "GB" if size else "inconnue")
            print("Type :", r.headers.get("Content-Type"))
    except Exception as e:
        print("Erreur :", str(e))

# Bibliothèque Scientifique — Nexus Science
## Phase 1 : Collecte et filtrage Open Library
Environnement cloud Kaggle — traitement en flux

In [ ]:
import os, gzip, json, urllib.request

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

print("Dossier de travail :", BASE)
print("Environnement prêt pour le traitement Open Library.")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
subjects = collections.Counter()
languages = collections.Counter()
years = collections.Counter()
count = 0
valid = 0

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            count += 1

            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                valid += 1

                for s in rec.get("subjects", []):
                    if isinstance(s, str):
                        subjects[s.lower()] += 1

                for lang in rec.get("languages", []):
                    if isinstance(lang, dict):
                        languages[lang.get("key", "")] += 1

                date = str(rec.get("publish_date", ""))
                if len(date) >= 4 and date[:4].isdigit():
                    years[date[:4]] += 1

            except Exception:
                pass

print("Lignes examinées :", count)
print("Notices JSON valides :", valid)
print("\nTOP 40 SUJETS :")
for s, n in subjects.most_common(40):
    print(n, "|", s)

print("\nTOP LANGUES :")
for s, n in languages.most_common(20):
    print(n, "|", s)

print("\nDATES les plus fréquentes :")
for s, n in years.most_common(20):
    print(n, "|", s)

In [ ]:
SCIENCE_KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "philosophy of science", "nursing", "veterinary"
]

print("Filtre scientifique initialisé.")
print("Nombre de mots-clés :", len(SCIENCE_KEYWORDS))
print("Période cible : 1720–2026")
print("Langues cibles : multilingues")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = 0
selected = 0
matched_subjects = collections.Counter()
examples = []

keywords = [k.lower() for k in SCIENCE_KEYWORDS]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = rec.get("subjects", [])
                subject_text = " ".join(
                    s.lower() for s in subjects if isinstance(s, str)
                )
                text_to_check = title + " " + subject_text

                if any(k in text_to_check for k in keywords):
                    selected += 1
                    for s in subjects:
                        if isinstance(s, str):
                            matched_subjects[s] += 1

                    if len(examples) < 15:
                        examples.append(rec.get("title", "Sans titre"))

            except Exception:
                pass

print("Notices examinées :", count)
print("Notices sélectionnées :", selected)
print("Proportion :", round(selected/count*100, 2) if count else 0, "%")
print("\nEXEMPLES :")
for title in examples:
    print("-", title)

print("\nSUJETS RETENUS :")
for subject, n in matched_subjects.most_common(20):
    print(n, "|", subject)

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = selected = excluded = 0
examples = []

science = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "medicine", "medical",
    "anatomy", "engineering", "computer science", "astronomy",
    "geology", "ecology", "botany", "zoology", "statistics",
    "microbiology", "genetics", "pharmacology", "neuroscience",
    "agriculture", "nursing", "veterinary", "technology",
    "environmental science", "biotechnology", "physics",
    "natural science", "research methodology"
]

noise = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "bill [", "regulations",
    "establishment order", "amendment order",
    "house of lords", "government notice"
]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = [
                    s.lower() for s in rec.get("subjects", [])
                    if isinstance(s, str)
                ]
                searchable = title + " " + " ".join(subjects)

                relevant = any(k in searchable for k in science)
                administrative = any(k in title for k in noise)

                if relevant and not administrative:
                    selected += 1
                    if len(examples) < 25:
                        examples.append(title)
                elif relevant and administrative:
                    excluded += 1

            except Exception:
                pass

print("Notices examinées :", count)
print("Retenues V2 :", selected)
print("Documents administratifs écartés :", excluded)
print("\nEXEMPLES RETENUS :")
for t in examples:
    print("-", t)

In [ ]:
# NEXUS SCIENCE — GLOBAL OPEN LIBRARY EDITIONS SCANNER V1
# Streaming + checkpoint + restart-safe
# Aucune écriture dans Neon.

import os, json, gzip, urllib.request, time, re

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

URL = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
OUT = os.path.join(BASE, "scientific_editions.jsonl")
CHECKPOINT = os.path.join(BASE, "scanner_checkpoint.json")

KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "nursing", "veterinary", "biotechnology", "mathematics",
    "matematica", "ciencia", "ciencias", "biologia",
    "fisica", "quimica", "medicina", "ingenieria",
    "astronomia", "mathematiques", "scientifique",
    "biologie", "physique", "chimie", "médecine",
    "matemática", "ciência", "biologia", "medicina",
    "engenharia", "astronomia"
]

# Termes fortement administratifs : exclusions prudentes sur le titre seulement
NOISE = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "government notice",
    "house of lords bill"
]

def matches(record):
    title = str(record.get("title", "")).lower()
    subjects = record.get("subjects", [])
    if not isinstance(subjects, list):
        subjects = []

    subject_text = " ".join(
        s.lower() for s in subjects if isinstance(s, str)
    )
    searchable = title + " " + subject_text

    relevant = any(k in searchable for k in KEYWORDS)
    excluded = any(k in title for k in NOISE)

    return relevant and not excluded

# Reprise du compteur de lignes
resume_line = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, "r", encoding="utf-8") as f:
        state = json.load(f)
        resume_line = state.get("scan_lines", 0)

print("Scanner Nexus Science")
print("Lignes déjà parcourues :", resume_line)
print("Fichier de sortie :", OUT)
print("Mode : flux gzip / checkpoint")

scan_lines = resume_line
valid = 0
selected = 0
started = time.time()
last_checkpoint = scan_lines

# append pour préserver les résultats précédents
with open(OUT, "a", encoding="utf-8") as output:
    request = urllib.request.Request(
        URL, headers={"User-Agent": "NexusScience-Research/1.0"}
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        with gzip.GzipFile(fileobj=response, mode="rb") as gz:

            current_line = 0

            for raw_line in gz:
                current_line += 1

                # Relecture du début pour reprendre le checkpoint
                if current_line <= resume_line:
                    if current_line % 1000000 == 0:
                        print("Reprise :", current_line)
                    continue

                scan_lines += 1

                parts = raw_line.rstrip(b"\n").split(b"\t")
                if len(parts) < 4:
                    continue

                try:
                    record = json.loads(parts[-1])
                    valid += 1

                    if matches(record):
                        output.write(json.dumps(
                            record, ensure_ascii=False
                        ) + "\n")
                        selected += 1

                except Exception:
                    pass

                if scan_lines - last_checkpoint >= 10000:
                    output.flush()

                    with open(CHECKPOINT, "w", encoding="utf-8") as cp:
                        json.dump(
                            {"scan_lines": scan_lines},
                            cp
                        )

                    last_checkpoint = scan_lines
                    elapsed = round(time.time() - started, 1)

                    print(
                        "Progression lignes :", scan_lines,
                        "| JSON valides :", valid,
                        "| retenues :", selected,
                        "| secondes :", elapsed
                    )

# Dernier checkpoint
with open(CHECKPOINT, "w", encoding="utf-8") as cp:
    json.dump({"scan_lines": scan_lines}, cp)

print("\nSCAN TERMINÉ")
print("Lignes parcourues :", scan_lines)
print("JSON valides :", valid)
print("Retenues pendant cette session :", selected)
print("Temps (sec) :", round(time.time() - started, 1))
print("Résultats :", OUT)

In [ ]:
!ls -lh /kaggle/working/nexus_science

In [ ]:
import json

with open("/kaggle/working/nexus_science/scanner_checkpoint.json", "r") as f:
    print(json.load(f))

In [ ]:
import os

p = "/kaggle/working/nexus_science/scientific_editions.jsonl"

print("Fichier présent :", os.path.exists(p))
print("Taille (Go) :", round(os.path.getsize(p) / 1024**3, 3))

with open(p, "rb") as f:
    f.seek(max(0, os.path.getsize(p) - 2000))
    print("\nFIN DU FICHIER :\n")
    print(f.read().decode("utf-8", errors="replace"))